In [1]:
import torch
import numpy as np
import pandas as pd
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import warnings
warnings.filterwarnings('ignore')

# Reload model
print("Loading GPT-2...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
tokenizer.pad_token = tokenizer.eos_token
print("Done.")

# Load results
df_full = pd.read_csv('results/ragtruth_results.csv')
labels = df_full['is_hallucinated'].values
print(f"Loaded {len(df_full)} samples.")

# SelfCheckGPT works by:
# 1. Generating N samples of the same answer from the model
# 2. Checking consistency between samples
# 3. Inconsistent answers = hallucination signal
# We implement a lightweight version using token probability consistency

def selfcheck_score(text, context, n_samples=5, max_length=150):
    """
    Lightweight SelfCheckGPT implementation.
    Generates n_samples continuations and measures consistency
    via average pairwise token probability divergence.
    """
    input_text = f"Context: {context[:400]}\n\nAnswer: {text[:200]}"
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True
    )
    
    all_probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits[0], dim=-1).numpy()
            all_probs.append(probs)
    
    # Measure inconsistency: std across samples at each token position
    min_len = min(p.shape[0] for p in all_probs)
    all_probs = np.array([p[:min_len] for p in all_probs])
    
    # Higher std = more inconsistent = more likely hallucinated
    inconsistency = float(np.mean(np.std(all_probs, axis=0)))
    return inconsistency

'''# Test on one sample
sample = df_full.iloc[0]
score = selfcheck_score(str(sample['output'])[:200], 
                        str(sample['context'])[:400])
print(f"\nSelfCheckGPT test score: {score:.6f}")
print("SelfCheckGPT baseline ready.")'''


d:\Nlp_Assignment\Nlp_Assignment\new_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading GPT-2...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3401.50it/s]


Done.
Loaded 500 samples.


'# Test on one sample\nsample = df_full.iloc[0]\nscore = selfcheck_score(str(sample[\'output\'])[:200], \n                        str(sample[\'context\'])[:400])\nprint(f"\nSelfCheckGPT test score: {score:.6f}")\nprint("SelfCheckGPT baseline ready.")'

In [2]:
print(df_full.columns.tolist())
print(df_full.head(2))



['entropy_only', 'information_gain', 'kl_divergence', 'confidence_drop', 'semantic_entropy', 'ig_per_token', 'entropy_per_token', 'is_hallucinated', 'halu_type', 'model', 'task_type', 'composite', 'composite_v2']
   entropy_only  information_gain  kl_divergence  confidence_drop  \
0      3.986445          0.594633      10.017441        -0.111249   
1      4.013071          0.789923       9.295832        -0.120514   

   semantic_entropy                                       ig_per_token  \
0          1.291537  [1.4844684600830078, -0.5094485282897949, 4.84...   
1          1.224328  [0.4764862060546875, -2.583496570587158, -1.28...   

                                   entropy_per_token  is_hallucinated  \
0  [6.674890518188477, 7.90794038772583, 3.885983...                1   
1  [6.674890518188477, 7.90794038772583, 3.885983...                1   

       halu_type             model task_type  composite  composite_v2  
0    unsupported  llama-2-70b-chat  Data2txt   5.663363      5.6

In [3]:
# Reload original RAGTruth data
df_rag_original = pd.read_csv('data/ragtruth.csv')
print("Original columns:", df_rag_original.columns.tolist())
print(f"Original shape: {df_rag_original.shape}")

Original columns: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str', 'is_hallucinated', 'halu_type', 'labels_parsed']
Original shape: (15090, 14)


In [4]:
import json

# Reconstruct which rows were used in our 500-sample run
# by matching on stratified sample with same random seed
df_halu_samples = df_rag_original[df_rag_original['is_hallucinated']==1].sample(250, random_state=42)
df_clean_samples = df_rag_original[df_rag_original['is_hallucinated']==0].sample(250, random_state=42)
df_sample_original = pd.concat([df_halu_samples, df_clean_samples]).reset_index(drop=True)

# Merge with results
df_merged = pd.concat([df_sample_original.reset_index(drop=True), 
                       df_full.reset_index(drop=True)], axis=1)
df_merged = df_merged.loc[:,~df_merged.columns.duplicated()]

print(f"Merged shape: {df_merged.shape}")
print("Columns:", df_merged.columns.tolist())
df_merged.to_csv('results/ragtruth_merged.csv', index=False)
print("Saved to results/ragtruth_merged.csv")

Merged shape: (500, 23)
Columns: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str', 'is_hallucinated', 'halu_type', 'labels_parsed', 'entropy_only', 'information_gain', 'kl_divergence', 'confidence_drop', 'semantic_entropy', 'ig_per_token', 'entropy_per_token', 'composite', 'composite_v2']
Saved to results/ragtruth_merged.csv


In [5]:
# Reload merged data with text columns
df_merged = pd.read_csv('results/ragtruth_merged.csv')
labels = df_merged['is_hallucinated'].values
print(f"Loaded {len(df_merged)} samples.")
print("Columns available:", df_merged.columns.tolist())

# Test selfcheck on one sample
sample = df_merged.iloc[0]
score = selfcheck_score(str(sample['output'])[:200], 
                        str(sample['context'])[:400])
print(f"\nSelfCheckGPT test score: {score:.6f}")
print("Ready to run on full dataset.")

Loaded 500 samples.
Columns available: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str', 'is_hallucinated', 'halu_type', 'labels_parsed', 'entropy_only', 'information_gain', 'kl_divergence', 'confidence_drop', 'semantic_entropy', 'ig_per_token', 'entropy_per_token', 'composite', 'composite_v2']

SelfCheckGPT test score: 0.000000
Ready to run on full dataset.


In [6]:
def selfcheck_score(text, context, n_samples=5, max_length=150):
    """
    Lightweight SelfCheckGPT with temperature sampling.
    """
    input_text = f"Context: {context[:400]}\n\nAnswer: {text[:200]}"
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True
    )

    all_probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            outputs = model(
                **inputs,
                temperature=1.5,  # add randomness
            )
            # Add small random noise to simulate sampling variance
            noise = torch.randn_like(outputs.logits) * 0.1
            noisy_logits = outputs.logits + noise
            probs = torch.softmax(noisy_logits[0], dim=-1).numpy()
            all_probs.append(probs)

    min_len = min(p.shape[0] for p in all_probs)
    all_probs = np.array([p[:min_len] for p in all_probs])

    # Higher std = more inconsistent = more likely hallucinated
    inconsistency = float(np.mean(np.std(all_probs, axis=0)))
    return inconsistency

# Test again
sample = df_merged.iloc[0]
score = selfcheck_score(str(sample['output'])[:200],
                        str(sample['context'])[:400])
print(f"SelfCheckGPT test score: {score:.6f}")

# Test on a few samples to see variation
print("\nScores on first 5 samples:")
for i in range(5):
    s = df_merged.iloc[i]
    sc = selfcheck_score(str(s['output'])[:200], str(s['context'])[:400])
    print(f"  Sample {i} (hallucinated={s['is_hallucinated']}): {sc:.6f}")

SelfCheckGPT test score: 0.000001

Scores on first 5 samples:
  Sample 0 (hallucinated=1): 0.000001
  Sample 1 (hallucinated=1): 0.000001
  Sample 2 (hallucinated=1): 0.000001
  Sample 3 (hallucinated=1): 0.000001
  Sample 4 (hallucinated=1): 0.000001


In [7]:
def selfcheck_score(text, context, max_length=150):
    """
    SelfCheckGPT-style consistency check using prompt variations.
    Measures how much the model's confidence changes across
    different phrasings of the same context.
    """
    # Create 3 prompt variations
    prompts = [
        f"Context: {context[:400]}\n\nAnswer: {text[:200]}",
        f"Based on the following: {context[:400]}\n\nResponse: {text[:200]}",
        f"Given: {context[:400]}\n\nThe answer is: {text[:200]}",
    ]
    
    all_probs = []
    for prompt in prompts:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            max_length=max_length,
            truncation=True
        )
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits[0], dim=-1).numpy()
            all_probs.append(probs)

    # Align lengths
    min_len = min(p.shape[0] for p in all_probs)
    all_probs = np.array([p[:min_len] for p in all_probs])

    # Measure inconsistency across prompt variations
    # Higher variance = model is sensitive to phrasing = less faithful
    token_vars = np.var(all_probs, axis=0)
    inconsistency = float(np.mean(np.max(token_vars, axis=1)))
    return inconsistency

# Test on first 5 samples
print("SelfCheckGPT scores on first 5 samples:")
for i in range(5):
    s = df_merged.iloc[i]
    sc = selfcheck_score(str(s['output'])[:200], str(s['context'])[:400])
    print(f"  Sample {i} (hallucinated={s['is_hallucinated']}): {sc:.8f}")

SelfCheckGPT scores on first 5 samples:
  Sample 0 (hallucinated=1): 0.13755822
  Sample 1 (hallucinated=1): 0.12457191
  Sample 2 (hallucinated=1): 0.09954926
  Sample 3 (hallucinated=1): 0.05486752
  Sample 4 (hallucinated=1): 0.13760689


In [8]:
import time

print("Running SelfCheckGPT on 500 samples...")
print("Estimated time: 10-15 minutes\n")

selfcheck_scores = []
start = time.time()

for i, row in df_merged.iterrows():
    try:
        score = selfcheck_score(
            str(row['output'])[:200],
            str(row['context'])[:400]
        )
        selfcheck_scores.append(score)
    except:
        selfcheck_scores.append(0.0)
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        remaining = (500 - (i + 1)) / rate
        print(f"  {i+1}/500 done — ~{remaining/60:.1f} minutes remaining")

df_merged['selfcheck_score'] = selfcheck_scores

# Evaluate
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

def compute_auroc_with_ci(scores, labels, n_bootstrap=1000):
    auroc = roc_auc_score(labels, scores)
    bootstrap_aurocs = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(labels), len(labels), replace=True)
        try:
            b_auroc = roc_auc_score(labels[idx], scores[idx])
            bootstrap_aurocs.append(b_auroc)
        except:
            continue
    ci_lower = np.percentile(bootstrap_aurocs, 2.5)
    ci_upper = np.percentile(bootstrap_aurocs, 97.5)
    return auroc, ci_lower, ci_upper

def compute_f1(scores, labels):
    thresholds = np.percentile(scores, np.arange(10, 90, 5))
    best_f1 = 0
    for thresh in thresholds:
        preds = (scores >= thresh).astype(int)
        from sklearn.metrics import f1_score
        f1 = f1_score(labels, preds, zero_division=0)
        best_f1 = max(best_f1, f1)
    return best_f1

def compute_ece(scores, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)
    for i in range(n_bins):
        mask = (scores_norm >= bins[i]) & (scores_norm < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = labels[mask].mean()
        bin_conf = scores_norm[mask].mean()
        ece += mask.sum() * abs(bin_acc - bin_conf)
    return ece / len(labels)

sc_scores = np.array(selfcheck_scores)
sc_labels = df_merged['is_hallucinated'].values

auroc, ci_low, ci_high = compute_auroc_with_ci(sc_scores, sc_labels)
f1 = compute_f1(sc_scores, sc_labels)
rho, _ = spearmanr(sc_scores, sc_labels)
ece = compute_ece(sc_scores, sc_labels)

print(f"\nSelfCheckGPT (Baseline 2) Results:")
print(f"  AUROC:    {auroc:.4f} [{ci_low:.3f}-{ci_high:.3f}]")
print(f"  F1:       {f1:.4f}")
print(f"  Spearman: {rho:.4f}")
print(f"  ECE:      {ece:.4f}")

df_merged.to_csv('results/ragtruth_merged.csv', index=False)
print("\nSaved.")

Running SelfCheckGPT on 500 samples...
Estimated time: 10-15 minutes

  50/500 done — ~5.8 minutes remaining
  100/500 done — ~5.2 minutes remaining
  150/500 done — ~4.6 minutes remaining
  200/500 done — ~3.9 minutes remaining
  250/500 done — ~3.3 minutes remaining
  300/500 done — ~2.6 minutes remaining
  350/500 done — ~2.0 minutes remaining
  400/500 done — ~1.3 minutes remaining
  450/500 done — ~0.6 minutes remaining
  500/500 done — ~0.0 minutes remaining

SelfCheckGPT (Baseline 2) Results:
  AUROC:    0.6861 [0.638-0.733]
  F1:       0.6708
  Spearman: 0.3223
  ECE:      0.1158

Saved.


In [9]:
import torch
import numpy as np
import pandas as pd
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import warnings
warnings.filterwarnings('ignore')

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
tokenizer.pad_token = tokenizer.eos_token
print("Model loaded.")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4182.77it/s]


Model loaded.


In [10]:
df_merged = pd.read_csv('results/ragtruth_merged.csv')
labels = df_merged['is_hallucinated'].values
print(f"Loaded {len(df_merged)} samples.")

Loaded 500 samples.


In [11]:
def selfcheck_score(text, context, max_length=150):
    prompts = [
        f"Context: {context[:400]}\n\nAnswer: {text[:200]}",
        f"Based on the following: {context[:400]}\n\nResponse: {text[:200]}",
        f"Given: {context[:400]}\n\nThe answer is: {text[:200]}",
    ]
    all_probs = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt",
                          max_length=max_length, truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits[0], dim=-1).numpy()
            all_probs.append(probs)
    min_len = min(p.shape[0] for p in all_probs)
    all_probs = np.array([p[:min_len] for p in all_probs])
    token_vars = np.var(all_probs, axis=0)
    inconsistency = float(np.mean(np.max(token_vars, axis=1)))
    return inconsistency

print("selfcheck_score function ready.")

selfcheck_score function ready.


In [12]:
import time
from sklearn.metrics import roc_auc_score, f1_score
from scipy.stats import spearmanr

print("Running SelfCheckGPT on 500 samples...")

selfcheck_scores = []
start = time.time()

for idx in range(len(df_merged)):
    row = df_merged.iloc[idx]
    try:
        score = selfcheck_score(
            str(row['output'])[:150],
            str(row['context'])[:300]
        )
        selfcheck_scores.append(score)
    except:
        selfcheck_scores.append(0.0)
    
    if (idx + 1) % 10 == 0:
        elapsed = time.time() - start
        rate = (idx + 1) / elapsed
        remaining = (500 - (idx + 1)) / rate
        print(f"  {idx+1}/500 — ~{remaining/60:.1f} min remaining", flush=True)

print("Done!")

Running SelfCheckGPT on 500 samples...
  10/500 — ~6.3 min remaining
  20/500 — ~6.0 min remaining
  30/500 — ~5.7 min remaining
  40/500 — ~5.5 min remaining
  50/500 — ~5.5 min remaining
  60/500 — ~5.3 min remaining
  70/500 — ~5.2 min remaining
  80/500 — ~5.1 min remaining
  90/500 — ~4.9 min remaining
  100/500 — ~4.8 min remaining
  110/500 — ~4.6 min remaining
  120/500 — ~4.5 min remaining
  130/500 — ~4.4 min remaining
  140/500 — ~4.2 min remaining
  150/500 — ~4.1 min remaining
  160/500 — ~4.0 min remaining
  170/500 — ~3.8 min remaining
  180/500 — ~3.7 min remaining
  190/500 — ~3.6 min remaining
  200/500 — ~3.5 min remaining
  210/500 — ~3.4 min remaining
  220/500 — ~3.3 min remaining
  230/500 — ~3.1 min remaining
  240/500 — ~3.0 min remaining
  250/500 — ~2.9 min remaining
  260/500 — ~2.8 min remaining
  270/500 — ~2.7 min remaining
  280/500 — ~2.6 min remaining
  290/500 — ~2.4 min remaining
  300/500 — ~2.3 min remaining
  310/500 — ~2.2 min remaining
  320/500

In [13]:
def compute_auroc_with_ci(scores, labels, n_bootstrap=1000):
    auroc = roc_auc_score(labels, scores)
    bootstrap_aurocs = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(labels), len(labels), replace=True)
        try:
            b_auroc = roc_auc_score(labels[idx], scores[idx])
            bootstrap_aurocs.append(b_auroc)
        except:
            continue
    ci_lower = np.percentile(bootstrap_aurocs, 2.5)
    ci_upper = np.percentile(bootstrap_aurocs, 97.5)
    return auroc, ci_lower, ci_upper

def compute_f1(scores, labels):
    thresholds = np.percentile(scores, np.arange(10, 90, 5))
    best_f1 = 0
    for thresh in thresholds:
        preds = (scores >= thresh).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        best_f1 = max(best_f1, f1)
    return best_f1

def compute_ece(scores, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)
    for i in range(n_bins):
        mask = (scores_norm >= bins[i]) & (scores_norm < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = labels[mask].mean()
        bin_conf = scores_norm[mask].mean()
        ece += mask.sum() * abs(bin_acc - bin_conf)
    return ece / len(labels)

sc_scores = np.array(selfcheck_scores)
sc_labels = df_merged['is_hallucinated'].values

auroc, ci_low, ci_high = compute_auroc_with_ci(sc_scores, sc_labels)
f1 = compute_f1(sc_scores, sc_labels)
rho, _ = spearmanr(sc_scores, sc_labels)
ece = compute_ece(sc_scores, sc_labels)

print(f"SelfCheckGPT (Baseline 2) Results:")
print(f"  AUROC:    {auroc:.4f} [{ci_low:.3f}-{ci_high:.3f}]")
print(f"  F1:       {f1:.4f}")
print(f"  Spearman: {rho:.4f}")
print(f"  ECE:      {ece:.4f}")

df_merged['selfcheck_score'] = sc_scores
df_merged.to_csv('results/ragtruth_merged.csv', index=False)
print("\nSaved.")

SelfCheckGPT (Baseline 2) Results:
  AUROC:    0.6805 [0.636-0.729]
  F1:       0.6714
  Spearman: 0.3127
  ECE:      0.0873

Saved.
